# 머신러닝 고급 개념 입문

이 노트북에서는 지도학습과 비지도학습, 모델 평가, 튜닝, 신경망 기초, TensorFlow 입문을 다룹니다.

## Scikit-learn 기초와 대표 모델 둘러보기

Scikit-learn은 데이터를 **학습용(`fit`)**과 **예측용(`predict`)**으로 나누어 다양한 머신러닝 모델을 실험하기 좋은 라이브러리입니다.

- **회귀(Regression)**: 집값, 매출, 온도처럼 연속적인 숫자를 예측합니다.
- **분류(Classification)**: 정상/불량, 스팸/정상처럼 범주를 예측합니다.
- **비지도학습(Unsupervised learning)**: 정답 없이 비슷한 데이터끼리 묶거나 구조를 찾습니다.

아래 예제는 모델 사용법에 집중하기 위해 간단한 합성 데이터를 사용합니다. 실제 프로젝트에서는 결측치 처리, 특성 선택, 스케일링, 교차검증도 함께 고려해야 합니다.

In [ ]:
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score

# 모델 간 비교를 위해 회귀와 분류 데이터를 각각 생성합니다.
X_reg, y_reg = make_regression(
    n_samples=200, n_features=4, noise=15, random_state=42
)
X_cls, y_cls = make_classification(
    n_samples=200, n_features=4, n_informative=3,
    n_redundant=0, random_state=42
)

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)
X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

print('회귀 데이터:', X_reg_train.shape, X_reg_test.shape)
print('분류 데이터:', X_cls_train.shape, X_cls_test.shape)

### 1. 회귀 모델: 숫자 값을 예측하기

회귀 모델은 입력 특성으로부터 연속적인 숫자를 예측합니다. `MAE`는 평균적으로 얼마나 빗나갔는지, `R²`는 실제 변동을 얼마나 설명하는지 보여줍니다.

- **LinearRegression**: 특성과 target의 선형 관계가 대략 맞을 때 빠르고 해석하기 쉽습니다.
- **DecisionTreeRegressor**: 조건을 나누는 규칙으로 예측합니다. `max_depth`를 제한하지 않으면 과적합하기 쉽습니다.
- **RandomForestRegressor**: 여러 결정 트리를 학습해 평균을 내므로, 단일 트리보다 안정적인 경우가 많습니다.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

regression_models = {
    '선형 회귀': LinearRegression(),
    '디시전 트리 회귀': DecisionTreeRegressor(max_depth=4, random_state=42),
    '랜덤 포레스트 회귀': RandomForestRegressor(
        n_estimators=100, max_depth=5, random_state=42, n_jobs=-1
    ),
}

for name, model in regression_models.items():
    model.fit(X_reg_train, y_reg_train)
    prediction = model.predict(X_reg_test)
    print(
        f'{name}: MAE={mean_absolute_error(y_reg_test, prediction):.2f}, '
        f'R²={r2_score(y_reg_test, prediction):.2f}'
    )

### 2. 분류 모델: 범주를 예측하기

분류 모델은 각 샘플이 어느 클래스에 속하는지 예측합니다. `accuracy`는 전체 예측 중 맞힌 비율입니다. 클래스 불균형이 심하면 정밀도, 재현율, F1 점수도 함께 확인해야 합니다.

- **LogisticRegression**: 이름에 회귀가 들어가지만 대표적인 이진/다중 분류 모델입니다. 선형 경계를 학습합니다.
- **DecisionTreeClassifier**: 사람이 읽을 수 있는 if-then 규칙에 가깝고, 전처리가 비교적 간단합니다.
- **RandomForestClassifier**: 여러 트리의 다수결을 사용해 튜닝을 처음 시작할 때 자주 쓰는 강력한 기준 모델입니다.
- **KNeighborsClassifier**: 주변에 있는 데이터의 클래스를 보고 결정합니다. 특성의 스케일에 민감합니다.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

classification_models = {
    '로지스틱 회귀': LogisticRegression(max_iter=1000),
    'K-최근접 이웃': KNeighborsClassifier(n_neighbors=5),
    '디시전 트리 분류': DecisionTreeClassifier(max_depth=4, random_state=42),
    '랜덤 포레스트 분류': RandomForestClassifier(
        n_estimators=100, max_depth=5, random_state=42, n_jobs=-1
    ),
}

for name, model in classification_models.items():
    model.fit(X_cls_train, y_cls_train)
    prediction = model.predict(X_cls_test)
    print(f'{name}: accuracy={accuracy_score(y_cls_test, prediction):.2f}')

### 2-1. 분류 모델 평가: 혼동행렬(Confusion Matrix)

분류 모델은 예측값만 보는 것보다 **실제 정답과 어떤 방식으로 틀렸는지**를 함께 봐야 합니다. 혼동행렬은 기본적으로 **행(row)이 실제값**, **열(column)이 예측값**입니다.

|  | 예측 0 | 예측 1 |
| --- | ---: | ---: |
| 실제 0 | **TN**: 정상인데 정상으로 예측 | **FP**: 정상인데 불량으로 예측 |
| 실제 1 | **FN**: 불량인데 정상으로 예측 | **TP**: 불량을 불량으로 예측 |

- **Accuracy** = 전체 중 맞힌 비율: `(TP + TN) / 전체`
- **Precision** = 불량이라고 예측한 것 중 실제 불량 비율: `TP / (TP + FP)`
- **Recall** = 실제 불량 중 찾아낸 비율: `TP / (TP + FN)`
- **F1-score** = precision과 recall의 조화평균

예를 들어 불량을 놓치는 비용이 크다면, accuracy보다 **recall**을 더 중요하게 봐야 할 수 있습니다.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
import matplotlib.pyplot as plt

# 앞에서 학습한 랜덤 포레스트 분류 모델의 예측을 평가합니다.
classifier = classification_models['랜덤 포레스트 분류']
prediction = classifier.predict(X_cls_test)

matrix = confusion_matrix(y_cls_test, prediction)
print('행 = 실제값, 열 = 예측값')
print(matrix)
print()
print(classification_report(y_cls_test, prediction, target_names=['정상(0)', '불량(1)']))

ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=['정상(0)', '불량(1)']
).plot(cmap='Blues')
plt.title('Random Forest Confusion Matrix')
plt.show()

### 2-2. Validation: K-Fold 교차검증

train/test 한 번만 나누면 우연히 어떤 데이터가 test에 들어갔는지에 따라 점수가 달라질 수 있습니다. **K-Fold 교차검증**은 학습 데이터를 K개로 나누고, 매번 한 조각을 validation으로 사용하면서 K번 평가합니다.

- **Training set**: 모델 학습에 사용합니다.
- **Validation set**: 모델과 하이퍼파라미터를 비교·선택하는 데 사용합니다. K-Fold에서는 training set 안에서 여러 번 바뀝니다.
- **Test set**: 최종 모델을 선택한 뒤, 마지막 성능 확인에 딱 한 번 사용합니다.

분류에서는 각 fold의 클래스 비율을 유지하는 `StratifiedKFold`를 주로 사용합니다. 회귀에서는 일반적인 `KFold`를 사용할 수 있습니다. 평균 점수뿐 아니라 fold별 점수의 차이도 보면 모델 성능의 안정성을 알 수 있습니다.

In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate

# 분류: 각 fold에서 클래스 비율을 유지하는 StratifiedKFold를 사용합니다.
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
classification_cv = cross_validate(
    RandomForestClassifier(
        n_estimators=100, max_depth=5, random_state=42, n_jobs=-1
    ),
    X_cls_train,
    y_cls_train,
    cv=stratified_cv,
    scoring=['accuracy', 'precision', 'recall', 'f1']
)

print('분류 validation accuracy:', classification_cv['test_accuracy'].round(3))
print('분류 accuracy 평균:', classification_cv['test_accuracy'].mean().round(3))
print('분류 accuracy 표준편차:', classification_cv['test_accuracy'].std().round(3))
print('분류 recall 평균:', classification_cv['test_recall'].mean().round(3))

# 회귀: 각 fold를 번갈아 validation으로 사용하는 KFold를 사용합니다.
regression_cv = KFold(n_splits=5, shuffle=True, random_state=42)
regression_scores = cross_validate(
    RandomForestRegressor(
        n_estimators=100, max_depth=5, random_state=42, n_jobs=-1
    ),
    X_reg_train,
    y_reg_train,
    cv=regression_cv,
    scoring=['neg_mean_absolute_error', 'r2']
)

# scikit-learn은 손실 점수가 클수록 좋도록 음수 MAE를 반환합니다.
mae_scores = -regression_scores['test_neg_mean_absolute_error']
print('회귀 validation MAE:', mae_scores.round(2))
print('회귀 MAE 평균:', mae_scores.mean().round(2))
print('회귀 R² 평균:', regression_scores['test_r2'].mean().round(3))

### 3. 비지도학습: 정답 없이 그룹 찾기

**KMeans**는 데이터를 `n_clusters`개의 그룹으로 나눕니다. 정답 label이 없는 고객 세분화나 센서 상태 그룹화에 사용할 수 있지만, 클러스터 수를 미리 정해야 하고 특성 스케일의 영향을 받습니다.

In [ ]:
from sklearn.cluster import KMeans

cluster_model = KMeans(n_clusters=3, n_init=10, random_state=42)
cluster_labels = cluster_model.fit_predict(X_cls)

print('각 데이터의 클러스터 label:', cluster_labels[:10])
print('클러스터 중심 개수:', cluster_model.cluster_centers_.shape[0])

### 모델을 고를 때의 빠른 기준

| 목적 | 먼저 시도할 모델 | 기억할 점 |
| --- | --- | --- |
| 연속값 예측 | `LinearRegression`, `RandomForestRegressor` | MAE/RMSE와 R²를 함께 확인 |
| 범주 예측 | `LogisticRegression`, `RandomForestClassifier` | accuracy만으로 판단하지 않기 |
| 설명 가능한 규칙 | `DecisionTree` | 트리 깊이로 과적합 제어 |
| 가까운 샘플 기반 예측 | `KNeighborsClassifier` | 스케일링이 중요 |
| 정답 없는 그룹화 | `KMeans` | 클러스터 수와 스케일에 주의 |

실무에서는 하나의 모델을 미리 정답이라고 정하기보다, 같은 train/test 분할과 같은 평가 지표로 여러 모델을 비교한 뒤 교차검증과 하이퍼파라미터 튜닝을 진행합니다.

In [ ]:
from sklearn.cluster import KMeans
import pandas as pd

df = pd.DataFrame({
    'temperature_c': [70, 72, 68, 74, 66, 76],
    'vibration_mm': [2.0, 2.5, 1.8, 2.8, 1.6, 3.1]
})
km = KMeans(n_clusters=2, random_state=42, n_init=10)
print(km.fit_predict(df))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

quality_df = pd.read_csv('../data/manufacturing_data.csv')
X = quality_df[['temperature_c', 'vibration_mm', 'pressure_bar', 'downtime_min']]
y = quality_df['defect_flag']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('accuracy:', accuracy_score(y_test, pred))